## Video Demonstration

    Link:-


In [ ]:
from pathlib import Path
import json
from IPython.display import HTML, Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "integration").exists():
    ROOT = Path(r"F:\SEM IV\lessons\DB\Project\CS432_Track1_Submission\Module_A")

INTEGRATION = ROOT / "integration"
EVIDENCE = ROOT / "evidence"
DATABASE = ROOT / "database"

artifact_paths = {
    "demo": EVIDENCE / "02_module_a_demo_output.json",
    "parity": EVIDENCE / "04_module_a_parity_output.json",
    "aggregate_benchmarks": EVIDENCE / "07_module_a_benchmark_results.json",
    "detailed_benchmark": EVIDENCE / "benchmark_detailed.json",
    "render_manifest": INTEGRATION / "bptree_v2" / "render_manifest.json",
}

loaded = {name: json.loads(path.read_text(encoding="utf-8")) for name, path in artifact_paths.items()}
print("Notebook root:", ROOT)
for name, path in artifact_paths.items():
    print(f"{name}: {path.exists()} -> {path}")


## Report Structure

This report follows a conventional technical-report flow:

1. **Problem Statement**
2. **Solution Overview**
3. **Implementation Details**
4. **B+ Tree Logic**
5. **Integration and Validation**
6. **Tree Visualization**
7. **Performance Analysis**
8. **Discussion of Results**
9. **Conclusion**

That structure is deliberate: it makes the report easier to evaluate against the assignment brief and easier to defend during a viva.


## 1. Problem Statement

The assignment requires more than a generic implementation of a B+ Tree. It asks for a small but complete database-style package built around a custom B+ Tree index. The submission must therefore demonstrate all of the following in a coherent way:

- a B+ Tree implemented from scratch in Python
- support for insertion, exact search, update, deletion, and range query
- a lightweight database abstraction around the structure
- a brute-force baseline for comparison
- reproducible performance evidence
- visualization of the tree structure
- documentation that explains both the design and the observed results

A second practical challenge is that the data structure should not remain an isolated classroom artifact. To make the submission stronger, the same B+ Tree should also be shown working on meaningful application-shaped paths. In this package, that application-shaped context is the BlindDrop project domain.


## 2. Solution Overview

The solution is organised in two layers.

### 2.1 Standalone Module A layer

This is the direct answer to the assignment brief:

| File | Purpose |
| --- | --- |
| `database/bplustree.py` | Core B+ Tree implementation |
| `database/bruteforce.py` | Linear baseline used for comparison |
| `database/table.py` | Table abstraction over an index implementation |
| `database/db_manager.py` | Lightweight database manager for multiple tables |

### 2.2 BlindDrop integration layer

This second layer keeps the core implementation intact while reusing it on realistic project-shaped access paths:

| File | Purpose |
| --- | --- |
| `integration/blinddrop_index_manager.py` | Posting-list wrapper and domain-specific indexes |
| `integration/blinddrop_index_demo.py` | Demonstration of real lookup and range-scan paths |
| `integration/db_index_parity_demo.py` | DB-authoritative parity, rollback, repair, and rebuild proof |
| `integration/benchmark_blinddrop_paths.py` | Domain benchmark on BlindDrop-shaped workloads |
| `integration/render_bptree_v2.py` | Renderer for all project-oriented tree visuals |

The design goal is not to replace the relational backend with Python. Instead, the report treats the relational or exported state as authoritative and the custom B+ Tree as the assignment-facing indexing engine.


## 3. Implementation Details

### 3.1 Core classes and responsibilities

| Component | Responsibility |
| --- | --- |
| `BPlusTree` | Ordered multi-way search tree with linked leaves |
| `BPlusTreeNode` | Stores keys, child pointers, leaf values, and leaf-to-leaf linkage |
| `BruteForceDB` | Simple linear structure for exact and range-scan comparison |
| `Table` | Uniform CRUD-style wrapper around either B+ Tree or brute-force storage |
| `DatabaseManager` | Creates and manages multiple tables |

### 3.2 Submission-to-requirement mapping

| Requirement area | Implementation choice |
| --- | --- |
| From-scratch tree | Custom `BPlusTree` logic in Python |
| Exact search / range query | `search()` and `range_query()` in the B+ Tree |
| Update / delete | `update()` and `delete()` with structural rebalancing |
| DB-style abstraction | `Table` and `DatabaseManager` |
| Baseline comparison | `BruteForceDB` |
| Visual evidence | Graphviz-based tree rendering |
| Benchmarks | Domain benchmark plus detailed benchmark |

### 3.3 Snapshot-driven inputs used in the package

The integration, parity proof, renderer, and domain benchmark all start from the exported backend snapshot:

`F:\SEM IV\lessons\DB\Project\Project_432\backend\database_export.json`

| Snapshot table | Rows |
| --- | ---: |
| vaults | 120 |
| inner_tokens | 300 |
| files | 500 |
| file_key_access | 800 |
| sessions | 200 |
| auth_attempts | 1,200 |
| download_logs | 900 |
| captcha_tracking | 200 |
| expiry_jobs | 120 |
| portfolio_entries | 400 |

Using the exported project snapshot makes the integration reproducible and keeps the submission tied to the actual BlindDrop domain.


## 4. B+ Tree Logic

The B+ Tree implementation in this package follows the standard idea of keeping all record values in the leaf level while internal nodes store separator keys that guide traversal.

### 4.1 Search logic

Exact lookup starts at the root and moves downward through internal nodes using ordered separator keys. Once the target leaf is reached, binary-search style positioning is used to locate the key inside that leaf.

### 4.2 Insertion logic

Insertion first descends to the correct leaf. If a node exceeds the maximum number of keys after insertion, it is split. For leaf splits, the new sibling receives the upper half of the keys and the leaf-level linked list is updated. For internal-node splits, a separator key is promoted upward and child pointers are redistributed.

### 4.3 Deletion logic

Deletion removes the key from the target leaf and then checks whether the node fell below the minimum key threshold. If necessary, the implementation borrows from a sibling or merges adjacent nodes, then recomputes parent separators so the tree remains valid.

### 4.4 Update logic

Update is intentionally simpler than insert or delete because it only replaces the value attached to an existing key. No restructuring is required when the key already exists.

### 4.5 Range-query logic

Range query is where the B+ Tree shows one of its biggest advantages. The tree first navigates quickly to the leaf that should contain the start key. From there, it scans forward through linked leaves, which avoids rescanning the whole dataset and makes ordered retrieval efficient.

### 4.6 Why linked leaves matter

The linked-leaf design is important for this assignment because it explains why range scans behave differently from brute-force scanning. Once the correct start leaf is located, the query can continue in sorted order through neighbouring leaves without returning to internal nodes for every next key.


## 5. Integration and Validation

### 5.1 BlindDrop-specific indexed paths

The integration layer maps the standalone B+ Tree to these four project-shaped paths:

| Path | Key shape | Value / posting | Meaning in the domain |
| --- | --- | --- | --- |
| Outer-token lookup | `outer_token` | `vault_id` | Public vault discovery |
| Expiry range scan | `expires_at_epoch` | `vault_id[]` | Maintenance and expiry windows |
| Vault-file range scan | `(vault_id, status, created_at_epoch)` | `file_id[]` | Ordered listing of files inside a vault |
| Auth-attempt timeline | `(session_id, attempt_time_epoch)` | `auth_attempt_id[]` | Session-oriented security history |

### 5.2 Demo execution on the exported snapshot

The packaged demo selects real targets from the exported snapshot and exercises the four paths above.

- outer-token lookup target: `8Y2bTu5` -> `3e448b76-bcbc-43ce-8582-6998368bc51f`
- expiry range window: `1772795981` to `1772803181` -> `1` matching posting list(s)
- vault-file range: vault `3e448b76-bcbc-43ce-8582-6998368bc51f` -> `1` active file posting list(s)
- session timeline: session `8603789c-8f8d-446e-b525-ea342eaeb8ac` -> `8` auth-attempt posting list(s)

### 5.3 DB-authoritative validation story

The parity proof strengthens the report because it explicitly answers what happens when the index and the authoritative state diverge.

| Seeded vaults | Seeded files | Seeded auth attempts | Final vaults | Final files | Final auth attempts |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 120 | 500 | 1200 | 121 | 501 | 1201 |

| Parity step | Outcome |
| --- | --- |
| `initial_state` | parity ok = `True` |
| `commit_vault_success` | parity ok = `True` |
| `commit_file_success` | parity ok = `True` |
| `commit_auth_attempt_success` | parity ok = `True` |
| `forced_index_failure_with_rollback` | rollback worked = `True`, failed vault visible = `False` |
| `manual_divergence_detection` | parity ok = `False` |
| `read_path_lazy_repair` | repaired = `True`, parity after repair = `True` |
| `full_rebuild_from_authoritative_db` | parity before rebuild = `False`, parity after rebuild = `True` |

This gives the report a much stronger engineering story than a pure demo because it shows not only successful reads, but also rollback behavior, divergence detection, lazy repair, and full rebuild.


## 6. Tree Visualization

The visualization layer exists to make the B+ Tree structure inspectable rather than merely described. The renderer in `integration/render_bptree_v2.py` generates **19 BlindDrop-oriented B+ Tree PNGs** from the exported backend snapshot and records the run in `integration/bptree_v2/render_manifest.json`.

### 6.1 Why visualization matters

The rendered trees make the following design points concrete:

- different project tables lead to different key shapes
- larger tables result in deeper or wider trees
- composite keys can still be organised and traversed cleanly within the same B+ Tree model
- leaf-link structure can be discussed visually during evaluation

### 6.2 Rendered index inventory

| Rendered index | Table | Inserted rows | Total nodes | Leaf nodes | Internal nodes | Height |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| `01_vaults__outer_token` | `vaults` | 120 | 55 | 41 | 14 | 4 |
| `02_vaults__status_expires` | `vaults` | 120 | 53 | 39 | 14 | 4 |
| `03_inner_tokens__token_hash` | `inner_tokens` | 300 | 142 | 103 | 39 | 5 |
| `04_inner_tokens__vault_status` | `inner_tokens` | 300 | 151 | 108 | 43 | 5 |
| `05_files__vault_status_created` | `files` | 500 | 224 | 166 | 58 | 5 |
| `06_files__drive_file_id` | `files` | 500 | 240 | 175 | 65 | 5 |
| `07_files__deleted_at` | `files` | 90 | 43 | 31 | 12 | 4 |
| `08_fka__file_token` | `file_key_access` | 800 | 369 | 273 | 96 | 5 |
| `09_fka__token` | `file_key_access` | 800 | 393 | 289 | 104 | 5 |
| `10_sessions__ip_created` | `sessions` | 200 | 94 | 69 | 25 | 4 |
| `11_auth__vault_time` | `auth_attempts` | 1200 | 553 | 406 | 147 | 6 |
| `12_auth__session_time_success` | `auth_attempts` | 1200 | 571 | 419 | 152 | 6 |
| `13_auth__time` | `auth_attempts` | 1200 | 545 | 401 | 144 | 6 |
| `14_downloads__file_time` | `download_logs` | 900 | 416 | 304 | 112 | 6 |
| `15_downloads__token_time` | `download_logs` | 900 | 423 | 315 | 108 | 5 |
| `16_captcha__session` | `captcha_tracking` | 200 | 97 | 71 | 26 | 4 |
| `17_expiry__processed_sched` | `expiry_jobs` | 120 | 53 | 40 | 13 | 4 |
| `18_portfolio__vault_owner_status` | `portfolio_entries` | 400 | 189 | 134 | 55 | 5 |
| `19_portfolio__integrity_hash` | `portfolio_entries` | 400 | 186 | 137 | 49 | 5 |

### 6.3 Display helper

The next code cell renders the tree gallery dynamically, while the following markdown section embeds all 19 rendered trees directly into the notebook.


In [ ]:
tree_groups = {
    "Vault indexes": ["01_vaults__outer_token", "02_vaults__status_expires"],
    "Inner-token indexes": ["03_inner_tokens__token_hash", "04_inner_tokens__vault_status"],
    "File and key-access indexes": ["05_files__vault_status_created", "06_files__drive_file_id", "07_files__deleted_at", "08_fka__file_token", "09_fka__token"],
    "Session and auth indexes": ["10_sessions__ip_created", "11_auth__vault_time", "12_auth__session_time_success", "13_auth__time"],
    "Audit and maintenance indexes": ["14_downloads__file_time", "15_downloads__token_time", "16_captcha__session", "17_expiry__processed_sched"],
    "Portfolio indexes": ["18_portfolio__vault_owner_status", "19_portfolio__integrity_hash"],
}

parts = []
stats = loaded["render_manifest"]["indexes"]
for title, slugs in tree_groups.items():
    parts.append(f"<h3>{title}</h3>")
    parts.append("<div style='display:flex; flex-wrap:wrap; gap:18px; margin-bottom:24px;'>")
    for slug in slugs:
        image_path = INTEGRATION / "bptree_v2" / f"{slug}.png"
        stat = next(item for item in stats if item["slug"] == slug)
        parts.append(
            "<div style='width:360px; border:1px solid #d1d5db; border-radius:10px; padding:10px; background:#fafafa;'>"
            f"<div style='font-weight:600; margin-bottom:6px;'>{slug}</div>"
            f"<div style='font-size:12px; color:#4b5563; margin-bottom:8px;'>table={stat['table']}, rows={stat['rowsInserted']}, height={stat['stats']['height']}</div>"
            f"<img src='{image_path.as_posix()}' style='width:100%; border:1px solid #e5e7eb;'/>"
            "</div>"
        )
    parts.append("</div>")

display(HTML(''.join(parts)))


## 7.4 Complete Rendered Tree Gallery

The following static gallery embeds **all 19 rendered BlindDrop-oriented B+ Tree PNGs** directly in the notebook so the visual package can be reviewed even without running any display cell.

### Vault indexes
<div style="display:flex; flex-wrap:wrap; gap:18px; margin-bottom:18px;">
  <div style="width:360px;"><div><b>01_vaults__outer_token</b></div><img src="integration/bptree_v2/01_vaults__outer_token.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>02_vaults__status_expires</b></div><img src="integration/bptree_v2/02_vaults__status_expires.png" style="width:100%; border:1px solid #ddd;"></div>
</div>

### Inner-token indexes
<div style="display:flex; flex-wrap:wrap; gap:18px; margin-bottom:18px;">
  <div style="width:360px;"><div><b>03_inner_tokens__token_hash</b></div><img src="integration/bptree_v2/03_inner_tokens__token_hash.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>04_inner_tokens__vault_status</b></div><img src="integration/bptree_v2/04_inner_tokens__vault_status.png" style="width:100%; border:1px solid #ddd;"></div>
</div>

### File and key-access indexes
<div style="display:flex; flex-wrap:wrap; gap:18px; margin-bottom:18px;">
  <div style="width:360px;"><div><b>05_files__vault_status_created</b></div><img src="integration/bptree_v2/05_files__vault_status_created.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>06_files__drive_file_id</b></div><img src="integration/bptree_v2/06_files__drive_file_id.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>07_files__deleted_at</b></div><img src="integration/bptree_v2/07_files__deleted_at.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>08_fka__file_token</b></div><img src="integration/bptree_v2/08_fka__file_token.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>09_fka__token</b></div><img src="integration/bptree_v2/09_fka__token.png" style="width:100%; border:1px solid #ddd;"></div>
</div>

### Session and auth indexes
<div style="display:flex; flex-wrap:wrap; gap:18px; margin-bottom:18px;">
  <div style="width:360px;"><div><b>10_sessions__ip_created</b></div><img src="integration/bptree_v2/10_sessions__ip_created.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>11_auth__vault_time</b></div><img src="integration/bptree_v2/11_auth__vault_time.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>12_auth__session_time_success</b></div><img src="integration/bptree_v2/12_auth__session_time_success.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>13_auth__time</b></div><img src="integration/bptree_v2/13_auth__time.png" style="width:100%; border:1px solid #ddd;"></div>
</div>

### Audit and maintenance indexes
<div style="display:flex; flex-wrap:wrap; gap:18px; margin-bottom:18px;">
  <div style="width:360px;"><div><b>14_downloads__file_time</b></div><img src="integration/bptree_v2/14_downloads__file_time.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>15_downloads__token_time</b></div><img src="integration/bptree_v2/15_downloads__token_time.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>16_captcha__session</b></div><img src="integration/bptree_v2/16_captcha__session.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>17_expiry__processed_sched</b></div><img src="integration/bptree_v2/17_expiry__processed_sched.png" style="width:100%; border:1px solid #ddd;"></div>
</div>

### Portfolio indexes
<div style="display:flex; flex-wrap:wrap; gap:18px; margin-bottom:18px;">
  <div style="width:360px;"><div><b>18_portfolio__vault_owner_status</b></div><img src="integration/bptree_v2/18_portfolio__vault_owner_status.png" style="width:100%; border:1px solid #ddd;"></div>
  <div style="width:360px;"><div><b>19_portfolio__integrity_hash</b></div><img src="integration/bptree_v2/19_portfolio__integrity_hash.png" style="width:100%; border:1px solid #ddd;"></div>
</div>


## 7. Performance Analysis

The benchmark package answers two different questions, so it is split into two layers.

### 7.1 Domain benchmark

This benchmark starts from the exported BlindDrop snapshot, amplifies it deterministically, and measures the four project-shaped access paths. It answers the question: **does the custom B+ Tree help on the same kinds of paths that matter in the project domain?**

### 7.2 Detailed benchmark

This benchmark measures insert, search, delete, range, mixed workload, speedup, and memory trends over a broader sweep. It answers the question: **does the underlying data structure behave like a real B+ Tree even outside the project-specific wrapper?**

### 7.3 Complexity expectations

| Operation | B+ Tree expectation | Brute-force expectation |
| --- | --- | --- |
| Insert | logarithmic navigation plus local restructuring | linear growth |
| Search | logarithmic navigation to a target leaf | linear scan |
| Delete | logarithmic navigation plus possible rebalance | linear scan |
| Range query | logarithmic start plus linked-leaf traversal | full scan |

The observed plots should therefore be read not only as raw performance measurements, but as evidence that the empirical behavior matches the expected scaling pattern.


## 7.1 Domain Benchmark Results

The domain benchmark contains **20 measured points** from **500** to **19500** rows.

| Path | Average speedup | Peak speedup |
| --- | ---: | ---: |
| Outer lookup | 17.6x | 44.8x |
| Expiry range | 36.7x | 73.2x |
| File range | 62.0x | 129.8x |
| Auth range | 8.4x | 14.8x |

### Checkpoint values from the packaged run

| Size | Outer B+ ms | Outer Brute ms | Expiry B+ ms | Expiry Brute ms | File B+ ms | File Brute ms | Auth B+ ms | Auth Brute ms |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 500 | 0.0082 | 0.0053 | 0.0069 | 0.0222 | 0.0047 | 0.0287 | 0.0053 | 0.0249 |
| 9,500 | 0.0107 | 0.1868 | 0.0119 | 0.8714 | 0.0195 | 1.0534 | 0.0770 | 0.6943 |
| 19,500 | 0.0125 | 0.2056 | 0.0160 | 0.9206 | 0.0177 | 1.2076 | 0.1048 | 1.0977 |

### Interpretation

- outer-token lookup stays nearly flat for the B+ Tree while brute-force lookup grows with size
- expiry and file range scans show larger gains because the tree can jump to the start key and then continue through linked leaves
- auth-range still improves clearly, though the gap is smaller than the file path because of the distribution of session timelines and posting-list usage


In [ ]:
domain_images = [
    INTEGRATION / "benchmark_path_dashboard.png",
    INTEGRATION / "benchmark_path_speedup.png",
    INTEGRATION / "benchmark_outer_lookup.png",
    INTEGRATION / "benchmark_expiry_range.png",
    INTEGRATION / "benchmark_file_range.png",
    INTEGRATION / "benchmark_auth_range.png",
]

for image_path in domain_images:
    display(Markdown(f"### {image_path.name}"))
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"Missing: {image_path}")


## 7.2 Detailed Benchmark Results

The detailed benchmark contains **22 measured points** from **500** to **21,500** rows, with **2 averaged runs per point**.

The selective range benchmark now uses **32 fixed-width windows** per dataset, with the width set to roughly **1% of the dataset size**. That makes the range graph a better reflection of practical ordered retrieval than a near full-table scan.

### Checkpoint values from the packaged run

| Size | Insert B+ ms | Insert Brute ms | Search B+ ms | Search Brute ms | Delete B+ ms | Delete Brute ms | Range B+ ms | Range Brute ms |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 500 | 0.568 | 4.237 | 0.078 | 1.731 | 0.118 | 0.509 | 0.070 | 0.584 |
| 10,500 | 29.630 | 1863.597 | 0.158 | 34.027 | 0.286 | 11.487 | 0.267 | 9.100 |
| 21,500 | 73.620 | 7957.374 | 0.263 | 66.457 | 0.418 | 26.491 | 0.614 | 18.198 |

### Mixed workload and memory checkpoints

At **21,000 mixed operations**, the B+ Tree completes the workload in **53.841 ms** versus **1889.153 ms** for the brute-force baseline, which is approximately **35.1x faster** in this packaged run.

The memory evidence is now split into three views so the report does not conflate retained size with allocator peaks:

- `benchmark_memory.png`: retained structure size after insert using a recursive object-size walk
- `benchmark_memory_peak_allocations.png`: `tracemalloc` peak Python allocations during the build phase
- `benchmark_memory_after_delete.png`: retained structure size after deleting **20%** of keys

Checkpoint values for retained structure size after insert:

- at **1,000 records**, B+ Tree retained size is **59.37 KB** versus **90.73 KB**
- at **21,000 records**, B+ Tree retained size is **1239.85 KB** versus **1891.66 KB**

Checkpoint values for peak Python allocations during build:

- at **1,000 records**, B+ Tree peak allocation is **37.48 KB** versus **63.63 KB**
- at **21,000 records**, B+ Tree peak allocation is **759.83 KB** versus **1317.70 KB**

In this Python implementation the brute-force baseline stays heavier because it stores one tuple object per row in a growing list, while the B+ Tree stores keys and values inside shared node lists. The report should therefore describe the observed result directly instead of assuming the tree must consume more memory.


In [ ]:
detailed_images = [
    EVIDENCE / "benchmark_operation_dashboard.png",
    EVIDENCE / "benchmark_insertion.png",
    EVIDENCE / "benchmark_search.png",
    EVIDENCE / "benchmark_deletion.png",
    EVIDENCE / "benchmark_range.png",
    EVIDENCE / "benchmark_random_workload.png",
    EVIDENCE / "benchmark_speedup.png",
    EVIDENCE / "benchmark_memory.png",
]

for image_path in detailed_images:
    display(Markdown(f"### {image_path.name}"))
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"Missing: {image_path}")


## 8. Discussion of Results

The combined results support several conclusions.

### 8.1 Main findings

- the B+ Tree behaves as expected for exact search and range query workloads
- the BlindDrop domain benchmark shows that the structure is useful on realistic composite-key paths, not only on synthetic integer keys
- the detailed benchmark confirms that insert, search, delete, and range-query scaling trends are consistently better than the brute-force baseline
- the parity proof shows that the report does not ignore operational concerns such as rollback and recovery
- the renderer makes the structure easy to inspect and defend visually

### 8.2 Why the two benchmark layers matter together

If the submission only included the generic benchmark, it would prove algorithmic behavior but not project relevance. If it only included the BlindDrop benchmark, it would prove project relevance but not enough of the general data-structure story. Using both layers is therefore stronger than either one alone.

### 8.3 Practical interpretation

From an educational database perspective, the report shows why B+ Trees are useful for ordered access patterns: they support fast navigation to a start point and then efficient sequential traversal. That is exactly why they are appropriate for expiry scans, ordered file listings, and timeline analysis.


## 9. Conclusion

### 9.1 Achievements

- implemented a from-scratch B+ Tree in Python
- wrapped it in a lightweight database abstraction
- implemented a brute-force baseline for fair comparison
- demonstrated exact lookup, update, delete, and range-query support
- integrated the same structure into BlindDrop-shaped paths using exported project data
- produced reproducible benchmark, parity, and visualization evidence

### 9.2 Findings

- the B+ Tree clearly outperforms the brute-force baseline on both domain-aligned and generic workloads as the dataset grows
- range-oriented workloads benefit especially from linked-leaf traversal
- the project-shaped integration makes the assignment more convincing than a purely synthetic example
- visual and parity evidence make the submission easier to defend technically

### 9.3 Challenges

- the standalone tree stores one value per key, so duplicate-key project paths required a posting-list wrapper
- keeping the integration academically clean required separating authoritative DB state from the assignment-facing index layer
- packaging benchmarks and visuals in a submission-ready way required regenerating artifacts from a reproducible exported snapshot rather than relying on ad hoc demo data

### 9.4 Future Improvements

- add persistence for the custom B+ Tree so the structure can be stored and restored directly from disk
- support richer record schemas and secondary-index management inside the database layer
- add more automated invariant checks for structural validation after complex delete workloads
- extend the visualization layer with zoom-friendly SVG output and annotated legends
- compare against additional indexing strategies beyond the brute-force baseline

